# House Price Prediction - Model Walkthrough

This notebook demonstrates the process of building a high-accuracy house price prediction model using the dataset from `archive (6).zip`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import os

%matplotlib inline

## 1. Data Loading

In [2]:
data_path = 'data.csv'
df = pd.read_csv(data_path)
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


## 2. Preprocessing & Feature Engineering

In [3]:
# Remove zero prices and extreme outliers (top 1%)
df = df[df['price'] > 0]
q_hi = df['price'].quantile(0.99)
df = df[df['price'] < q_hi]

# Date features
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# New features
df['house_age'] = df['year'] - df['yr_built']
df['total_sqft'] = df['sqft_living'] + df['sqft_above'] + df['sqft_basement']

# Categorical Encoding
le_city = LabelEncoder()
df['city_encoded'] = le_city.fit_transform(df['city'])

le_zip = LabelEncoder()
df['zip_encoded'] = le_zip.fit_transform(df['statezip'])

df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,street,city,statezip,country,year,month,house_age,total_sqft,city_encoded,zip_encoded
0,2014-05-02,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,...,18810 Densmore Ave N,Shoreline,WA 98133,USA,2014,5,59,2680,36,62
2,2014-05-02,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,...,26206-26214 143rd Ave SE,Kent,WA 98042,USA,2014,5,48,3860,18,26
3,2014-05-02,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,...,857 170th Pl NE,Bellevue,WA 98008,USA,2014,5,51,4000,3,7
4,2014-05-02,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,...,9105 170th Ave NE,Redmond,WA 98052,USA,2014,5,38,3880,31,31
5,2014-05-02,490000.0,2.0,1.00,880,6380,1.0,0,0,3,...,522 NE 88th St,Seattle,WA 98115,USA,2014,5,76,1760,35,54


## 3. Training the Model

We use log transformation for the target variable `price` to improve accuracy.

In [4]:
features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 
            'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 
            'yr_built', 'yr_renovated', 'year', 'month', 'house_age', 
            'total_sqft', 'city_encoded', 'zip_encoded']

X = df[features]
y = np.log1p(df['price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

,loss,'squared_error'
,quantile,None
,learning_rate,0.05
,max_iter,1000
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'


## 4. Evaluation

In [5]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test)

print(f'R2 Score: {r2_score(y_test_orig, y_pred):.4f}')
print(f'MAE: ${mean_absolute_error(y_test_orig, y_pred):,.2f}')
print(f'RMSE: ${np.sqrt(mean_squared_error(y_test_orig, y_pred)):,.2f}')

R2 Score: 0.7432
MAE: $87,353.01
RMSE: $141,433.65
